# Quickstart
The tutorial will show you how to:  
- initialize a DNAStream file. 
- how to connect and stream the data in both read and append mode. 
- how to access built-in registries and provenance logs. 
- how to insert entities into a registry. 
- how to view records in a registry or provenance dataset. 
- how to extract the datasets as dataframes. 

In [ ]:
# load packages
from pathlib import Path
import tempfile
from dnastream import DNAStream

tmpdir = Path(tempfile.mkdtemp(prefix="dnastream_tutorial_"))
myfile = tmpdir / "temp.h5"


## DNAStream file initialization
As an added safety feature, `dnastream` will raise an exception if the path to the file being created already exists. 

In [ ]:
# Create the DNAStream file (fails if it already exists)
ds = DNAStream.create(path=str(myfile), patient_id="patientX", verbose=True)
ds.close()

print("Created:", myfile)

2026-02-05 13:01:54 - INFO - Created DNAStream file /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_myte69fp/temp.h5
2026-02-05 13:01:54 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_myte69fp/temp.h5' closed.


Created: /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_myte69fp/temp.h5


## Streaming a DNAStream file
There are two available modes for streaming a DNAStream file, 'r' and 'r+'.   
- 'r' allows read access only and prevents modifications to the DNAStream file
- 'r+' allows read/write access to facilitate modfications to the DNAStream file

It is reccomended to stream a DNAStream file using a context manager. See example below.

In [12]:
with DNAStream.open(path=myfile, mode="r", verbose=True) as ds:
    print(ds)

2026-02-04 13:05:03 - INFO - Connection to DNAStream file 'myfile.h5' open.
2026-02-04 13:05:03 - INFO - Connection to DNAStream file 'myfile.h5' closed.


DNAStream for Patient patientX with Package Version: 0.1.0b0
File path: myfile.h5
Created by: llweber on host LM4FNLXK2L at 2026-02-04T18:49:55.440044Z
Mode: 'r'
Dataset groups:
	'canonical': 0 tables
	'links': 0 tables
	'measurements': 0 tables
	'provenance': 1 tables
	'registry': 3 tables
	'results': 0 tables



## Accessing built-in objects, such as registry and provenance objects
DNStream comes with built-in registries to register and track `sample`, `variant` and `snp` entities and a provenance `log` to track all modifications to the DNAStream file. 

In [13]:
#obtain a reference to these build in objects
with DNAStream.open(path=myfile, mode="r") as ds:
    variant_registry = ds.variant
    print(f"Variant registry has {len(variant_registry)} entities.")
    sample_registry = ds.sample
    print(f"Sample registry has {len(sample_registry)} entities.")
    print(f"SNP registry has {len(ds.snp)} entities.")

    print(f"The provenance log contains {len(ds.log)} events.")


Variant registry has 0 entities.
Sample registry has 0 entities.
SNP registry has 0 entities.
The provenance log contains 4 events.


## Adding entities to the registry
Open the DNAStream in mode 'r+'. Then pass `Registry.add(...)` a list of dictionaries. The key is the name of the field and value is the value of the field.  See the built-in schemas for field names. Note that only required field must be included in the dictionary for each entity. In this example, `sample_name` is required but `modality` is optional. 

In [17]:
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample

    reg.add([
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ])

    print(f"Sample registry has {len(reg)} entities")

Sample registry has 2 entities


Entities an be added to the variant/snp registeries in an identical fashion.

In [18]:
with DNAStream.open(myfile, mode="r+") as ds:
       
    ds.variant.add([
        {"chrom": "chr1", "start_pos": 1231, "end_pos": 1232, "ref_allele": "A", "alt_allele": "T"},
    ])


    print(f"Variant registry has {len(ds.variant)} entities")

Variant registry has 1 entities


Since both Registry and Provenance objects are just interfaces to a 1-d dataset in an HDF5 file, iterating through each row of the dataset will yield a dictionary (key: field name, value: stored value) of the entitiy.

In [24]:
with DNAStream.open(myfile, mode="r") as ds:
    for snv in ds.variant:
        print(f"SNV label: {snv['label']} SNV id {snv['id']}  active: {snv['active']}")

       

SNV label: chr1:1231:A:T SNV id 0cc203f1-14a8-4548-ab48-ce90a254bb41  active: True


## Extracting HDF5 datasets to `pandas` dataframes
Registry and provenance objects can be extracted as `pandas.Dataframe`. Note that this provides a copy of the dataset. Modifications of this dataframe **WILL NOT MODIFY** the underlying HDF5 dataset. 

In [32]:
with DNAStream.open(myfile, mode="r") as ds:
    event_log_df = ds.log.to_dataframe()
    print(event_log_df)

                                     id                    timestamp     user  \
0  59f5fe9a-4fc9-4b7f-9742-61a6cb4b256d  2026-02-04T18:49:55.444453Z  llweber   
1  2754c45b-6791-4678-a1ac-5ac68312b329  2026-02-04T18:49:55.447316Z  llweber   
2  46b6e151-d9ca-49d7-a225-ed5232be7121  2026-02-04T18:49:55.448656Z  llweber   
3  16554999-a006-4d2a-aff3-97d5fd0d0c41  2026-02-04T18:49:55.453104Z  llweber   
4  6302c9a0-97e8-435f-8926-9ff45d393cba  2026-02-04T19:15:06.907695Z  llweber   
5  f644ca18-871a-4a54-af9b-3f05924c5cf3  2026-02-04T19:16:34.428933Z  llweber   

       scope   event            dataset                                source  \
0  DNAStream  create   /registry/sample    dnastream.registry.Registry.create   
1  DNAStream  create  /registry/variant    dnastream.registry.Registry.create   
2  DNAStream  create      /registry/snp    dnastream.registry.Registry.create   
3  DNAStream  create                  .  dnastream.dnastream.DNAStream.create   
4   registry  append   /reg

In [31]:
with DNAStream.open(myfile, mode="r") as ds:
    sample_df = ds.sample.to_dataframe()
    print(sample_df)

                                     id label  idx  active  \
0  fd6db04f-ad01-4041-abeb-28ee9d5345fa    S1    0    True   
1  a5f88fe5-1524-4a8c-a8b5-2dea2f36dc54    S2    1    True   

                    created_at created_by                  modified_at  \
0  2026-02-04T19:15:06.907378Z    llweber  2026-02-04T19:15:06.907378Z   
1  2026-02-04T19:15:06.907378Z    llweber  2026-02-04T19:15:06.907378Z   

  modified_by sample_name organism  ... center_name run study coverage  \
0     llweber          S1           ...                            NaN   
1     llweber          S2           ...                            NaN   

      modality location bam_file_path batch_id reference_build  \
0         bulk                                                   
1  single-cell                                                   

   date_of_sequencing  
0                      
1                      

[2 rows x 26 columns]
